In [53]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [54]:
from ucimlrepo import fetch_ucirepo

# Завантажуємо UCI Adult
adult = fetch_ucirepo(id=2)

# Об'єднуємо features і target в одну таблицю
df = pd.concat(
    [adult.data.features, adult.data.targets],
    axis=1
)

# Зберігаємо контрольований CSV
csv_path = '/content/hw3_adult.csv'
df.to_csv(csv_path, index=False)

print(df.shape)
display(df.head())

(48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [55]:
create_staging = """
DROP TABLE IF EXISTS hw3_adult CASCADE;
DROP TABLE IF EXISTS hw3_adult_staging CASCADE;

CREATE TABLE hw3_adult_staging (
    age_raw TEXT,
    workclass_raw TEXT,
    fnlwgt_raw TEXT,
    education_raw TEXT,
    education_num_raw TEXT,
    marital_status_raw TEXT,
    occupation_raw TEXT,
    relationship_raw TEXT,
    race_raw TEXT,
    sex_raw TEXT,
    capital_gain_raw TEXT,
    capital_loss_raw TEXT,
    hours_per_week_raw TEXT,
    native_country_raw TEXT,
    income_raw TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(create_staging))

print("Staging table created")

Staging table created


In [56]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_adult_staging (
                    age_raw,
                    workclass_raw,
                    fnlwgt_raw,
                    education_raw,
                    education_num_raw,
                    marital_status_raw,
                    occupation_raw,
                    relationship_raw,
                    race_raw,
                    sex_raw,
                    capital_gain_raw,
                    capital_loss_raw,
                    hours_per_week_raw,
                    native_country_raw,
                    income_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

pd.read_sql(
    'SELECT COUNT(*) AS loaded_rows FROM hw3_adult_staging',
    engine
)

,loaded_rows
0,48842


In [57]:
create_clean = """
DROP TABLE IF EXISTS hw3_adult CASCADE;

CREATE TABLE hw3_adult (
    person_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,

    age SMALLINT NOT NULL,
    workclass TEXT,
    fnlwgt INTEGER NOT NULL,
    education TEXT NOT NULL,
    education_num SMALLINT NOT NULL,
    marital_status TEXT NOT NULL,
    occupation TEXT,
    relationship TEXT NOT NULL,
    race TEXT NOT NULL,
    sex TEXT NOT NULL,
    capital_gain INTEGER NOT NULL,
    capital_loss INTEGER NOT NULL,
    hours_per_week SMALLINT NOT NULL,
    native_country TEXT,
    income TEXT NOT NULL,

    CHECK (age BETWEEN 16 AND 100),
    CHECK (hours_per_week BETWEEN 1 AND 168),
    CHECK (capital_gain >= 0),
    CHECK (capital_loss >= 0)
);
"""

with engine.begin() as conn:
    conn.execute(text(create_clean))

print("Clean table created")

Clean table created


In [58]:
insert_clean = """
INSERT INTO hw3_adult (
    age,
    workclass,
    fnlwgt,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income
)
SELECT
    TRIM(age_raw)::SMALLINT,
    NULLIF(NULLIF(TRIM(workclass_raw), ''), '?'),
    TRIM(fnlwgt_raw)::INTEGER,
    TRIM(education_raw),
    TRIM(education_num_raw)::SMALLINT,
    TRIM(marital_status_raw),
    NULLIF(NULLIF(TRIM(occupation_raw), ''), '?'),
    TRIM(relationship_raw),
    TRIM(race_raw),
    TRIM(sex_raw),
    TRIM(capital_gain_raw)::INTEGER,
    TRIM(capital_loss_raw)::INTEGER,
    TRIM(hours_per_week_raw)::SMALLINT,
    NULLIF(NULLIF(TRIM(native_country_raw), ''), '?'),
    RTRIM(TRIM(income_raw), '.')
FROM hw3_adult_staging;
"""

with engine.begin() as conn:
    conn.execute(text(insert_clean))

pd.read_sql(
    'SELECT COUNT(*) AS clean_rows FROM hw3_adult',
    engine
)

,clean_rows
0,48842


### Типізація та очищення даних

У staging-таблиці всі колонки зберігалися як TEXT, щоб сирі значення могли бути завантажені без помилок типізації.

У фінальній таблиці:

- `age` — SMALLINT, оскільки вік є невеликим цілим числом.
- `fnlwgt` — INTEGER, оскільки це числова вага запису в Census.
- `education_num` — SMALLINT, оскільки це числовий код рівня освіти.
- `capital_gain` та `capital_loss` — INTEGER, оскільки це невід'ємні цілі числові значення.
- `hours_per_week` — SMALLINT, оскільки кількість робочих годин є невеликим цілим числом.
- категоріальні ознаки (`workclass`, `education`, `marital_status`, `occupation`, `relationship`, `race`, `sex`, `native_country`, `income`) збережені як TEXT.

Для `workclass`, `occupation` та `native_country` значення `?` і порожні рядки були перетворені на SQL NULL.

Додатково встановлено перевірки допустимих значень для віку, кількості робочих годин, capital gain та capital loss.

In [59]:
row_count_query = """
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_adult_staging

UNION ALL

SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_adult;
"""

pd.read_sql(row_count_query, engine)

,layer,n_rows
0,staging,48842
1,clean,48842


### Висновок

Кількість рядків у staging- та clean-таблицях однакова — 48 842.

Під час очищення жоден рядок не був видалений. Значення `?` і порожні рядки в окремих категоріальних колонках були перетворені на SQL NULL, але самі записи залишилися в таблиці.

In [60]:
duplicates_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        age,
        workclass,
        fnlwgt,
        education,
        education_num,
        marital_status,
        occupation,
        relationship,
        race,
        sex,
        capital_gain,
        capital_loss,
        hours_per_week,
        native_country,
        income
    )) AS distinct_rows,
    COUNT(*) - COUNT(DISTINCT (
        age,
        workclass,
        fnlwgt,
        education,
        education_num,
        marital_status,
        occupation,
        relationship,
        race,
        sex,
        capital_gain,
        capital_loss,
        hours_per_week,
        native_country,
        income
    )) AS duplicate_rows
FROM hw3_adult;
"""

pd.read_sql(duplicates_query, engine)

,total_rows,distinct_rows,duplicate_rows
0,48842,48790,52


### Висновок

У наборі виявлено 29 повністю однакових записів за всіма змістовними колонками.

Оскільки датасет Adult не містить унікального ідентифікатора людини, ці записи не можна однозначно вважати помилковими дублікатами. Вони можуть належати різним людям з однаковими характеристиками, тому на цьому етапі записи не видаляються.

In [61]:
null_query = """
SELECT
    ROUND(
        COUNT(*) FILTER (WHERE workclass IS NULL) * 100.0 / COUNT(*),
        2
    ) AS pct_null_workclass,

    ROUND(
        COUNT(*) FILTER (WHERE occupation IS NULL) * 100.0 / COUNT(*),
        2
    ) AS pct_null_occupation,

    ROUND(
        COUNT(*) FILTER (WHERE native_country IS NULL) * 100.0 / COUNT(*),
        2
    ) AS pct_null_native_country,

    ROUND(
        COUNT(*) FILTER (WHERE education IS NULL) * 100.0 / COUNT(*),
        2
    ) AS pct_null_education,

    ROUND(
        COUNT(*) FILTER (WHERE income IS NULL) * 100.0 / COUNT(*),
        2
    ) AS pct_null_income,

    COUNT(*) AS total_rows

FROM hw3_adult;
"""

pd.read_sql(null_query, engine)

,pct_null_workclass,pct_null_occupation,pct_null_native_country,pct_null_education,pct_null_income,total_rows
0,5.73,5.75,1.75,0.0,0.0,48842


### Висновок

Найбільша частка пропусків спостерігається в `occupation` — 5.75% та `workclass` — 5.73%.

У `native_country` пропущено 1.75% значень. У `education` та `income` пропусків не виявлено.

Ці NULL з'явилися після нормалізації спеціального маркера `?` та порожніх рядків у сирих даних.

In [62]:
sentinel_query = """
SELECT
    COUNT(*) FILTER (
        WHERE TRIM(workclass_raw) = '?'
    ) AS question_workclass,

    COUNT(*) FILTER (
        WHERE TRIM(occupation_raw) = '?'
    ) AS question_occupation,

    COUNT(*) FILTER (
        WHERE TRIM(native_country_raw) = '?'
    ) AS question_native_country,

    COUNT(*) FILTER (
        WHERE TRIM(education_raw) = '?'
    ) AS question_education,

    COUNT(*) FILTER (
        WHERE TRIM(income_raw) = '?'
    ) AS question_income

FROM hw3_adult_staging;
"""

pd.read_sql(sentinel_query, engine)

,question_workclass,question_occupation,question_native_country,question_education,question_income
0,1836,1843,583,0,0


In [63]:
empty_query = """
SELECT
    COUNT(*) FILTER (
        WHERE TRIM(workclass_raw) = ''
    ) AS empty_workclass,

    COUNT(*) FILTER (
        WHERE TRIM(occupation_raw) = ''
    ) AS empty_occupation,

    COUNT(*) FILTER (
        WHERE TRIM(native_country_raw) = ''
    ) AS empty_native_country,

    COUNT(*) FILTER (
        WHERE TRIM(education_raw) = ''
    ) AS empty_education,

    COUNT(*) FILTER (
        WHERE TRIM(income_raw) = ''
    ) AS empty_income

FROM hw3_adult_staging;
"""

pd.read_sql(empty_query, engine)

,empty_workclass,empty_occupation,empty_native_country,empty_education,empty_income
0,0,0,0,0,0


In [64]:
staging_missing_query = """
SELECT
    COUNT(*) FILTER (WHERE workclass_raw IS NULL) AS null_workclass,
    COUNT(*) FILTER (WHERE TRIM(workclass_raw) = '?') AS question_workclass,

    COUNT(*) FILTER (WHERE occupation_raw IS NULL) AS null_occupation,
    COUNT(*) FILTER (WHERE TRIM(occupation_raw) = '?') AS question_occupation,

    COUNT(*) FILTER (WHERE native_country_raw IS NULL) AS null_native_country,
    COUNT(*) FILTER (WHERE TRIM(native_country_raw) = '?') AS question_native_country

FROM hw3_adult_staging;
"""

pd.read_sql(staging_missing_query, engine)

,null_workclass,question_workclass,null_occupation,question_occupation,null_native_country,question_native_country
0,963,1836,966,1843,274,583


### Висновок

У сирих даних пропуски представлені двома способами: SQL NULL та спеціальним маркером `?`.

Для `workclass` виявлено 963 NULL і 1836 значень `?`, для `occupation` — 966 NULL і 1843 `?`, для `native_country` — 274 NULL і 583 `?`.

Під час очищення обидва типи пропусків були приведені до SQL NULL.

In [65]:
income_distribution_query = """
SELECT
    income,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_adult
GROUP BY income
ORDER BY n_rows DESC;
"""

pd.read_sql(income_distribution_query, engine)

,income,n_rows,pct_rows
0,<=50K,37155,76.07
1,>50K,11687,23.93


### Висновок

У датасеті 76.07% записів належать до категорії доходу `<=50K`, а 23.93% — до категорії `>50K`.

Отже, цільова змінна `income` є незбалансованою: категорія `<=50K` зустрічається приблизно у три рази частіше. Це варто враховувати при подальшому використанні даних для задачі класифікації.

In [66]:
age_stats_query = """
SELECT
    COUNT(age) AS n_non_null,
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    ROUND(AVG(age)::NUMERIC, 2) AS avg_age,
    ROUND(STDDEV_SAMP(age)::NUMERIC, 2) AS sd_age
FROM hw3_adult;
"""

pd.read_sql(age_stats_query, engine)

,n_non_null,min_age,max_age,avg_age,sd_age
0,48842,17,90,38.64,13.71


### Висновок

У колонці `age` немає пропущених значень. Вік у датасеті знаходиться в межах від 17 до 90 років, середній вік становить 38.64 року, стандартне відхилення — 13.71.

Мінімальне та максимальне значення не суперечать допустимому діапазону, заданому для цієї колонки.

In [67]:
age_outlier_query = """
WITH stats AS (
    SELECT
        AVG(age) AS mean_age,
        STDDEV_SAMP(age) AS sd_age
    FROM hw3_adult
)
SELECT
    person_id,
    age,
    education,
    occupation,
    income,
    ROUND(
        ((age - mean_age) / NULLIF(sd_age, 0))::NUMERIC,
        2
    ) AS z_score
FROM hw3_adult
CROSS JOIN stats
WHERE ABS(age - mean_age) > 3 * sd_age
ORDER BY age DESC
LIMIT 20;
"""

pd.read_sql(age_outlier_query, engine)

,person_id,age,education,occupation,income,z_score
0,11515,90,HS-grad,Other-service,<=50K,3.75
1,10548,90,HS-grad,Craft-repair,>50K,3.75
2,8976,90,Bachelors,Sales,>50K,3.75
3,6236,90,Bachelors,Prof-specialty,<=50K,3.75
4,8966,90,HS-grad,None,<=50K,3.75
5,10213,90,Some-college,Farming-fishing,<=50K,3.75
6,2307,90,Some-college,Other-service,<=50K,3.75
7,5374,90,Masters,Exec-managerial,>50K,3.75
8,2896,90,Some-college,Adm-clerical,<=50K,3.75
9,223,90,HS-grad,Other-service,<=50K,3.75


### Висновок

Перевірка за правилом 3σ виявила записи з віком 90 років, для яких z-score становить приблизно 3.75.

Такі значення є статистично рідкісними, але не суперечать предметній області та можуть бути реальними. Тому ці записи залишаються в датасеті без змін.

## 5. DQL/EDA-запити

### Запит 1. Найпоширеніші типи зайнятості

**Business-question:** Які типи зайнятості (`workclass`) найпоширеніші в датасеті Adult?

In [68]:
query_1 = """
SELECT
    COALESCE(workclass, 'Unknown') AS workclass,
    COUNT(*) AS n_rows
FROM hw3_adult
GROUP BY COALESCE(workclass, 'Unknown')
ORDER BY n_rows DESC
LIMIT 10;
"""

pd.read_sql(query_1, engine)

,workclass,n_rows
0,Private,33906
1,Self-emp-not-inc,3862
2,Local-gov,3136
3,Unknown,2799
4,State-gov,1981
5,Self-emp-inc,1695
6,Federal-gov,1432
7,Without-pay,21
8,Never-worked,10


**Interpretation:** Найпоширеніший тип зайнятості — `Private`, він представлений 33 906 записами та суттєво переважає інші категорії.

Для 2 799 записів значення `workclass` відсутнє, тому за допомогою `COALESCE` вони відображені як `Unknown`. Найрідше зустрічаються категорії `Without-pay` та `Never-worked`.

### Запит 2. Дохід залежно від рівня освіти

**Business-question:** Яка частка людей з доходом `>50K` у різних категоріях освіти?

In [69]:
query_2 = """
SELECT
    education,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE income = '>50K') AS high_income_rows,
    ROUND(
        COUNT(*) FILTER (WHERE income = '>50K') * 100.0 / COUNT(*),
        2
    ) AS pct_high_income
FROM hw3_adult
GROUP BY education
ORDER BY pct_high_income DESC, n_rows DESC;
"""

pd.read_sql(query_2, engine)

,education,n_rows,high_income_rows,pct_high_income
0,Prof-school,834,617,73.98
1,Doctorate,594,431,72.56
2,Masters,2657,1459,54.91
3,Bachelors,8025,3313,41.28
4,Assoc-acdm,1601,413,25.80
5,Assoc-voc,2061,522,25.33
6,Some-college,10878,2063,18.96
7,HS-grad,15784,2503,15.86
8,12th,657,48,7.31
9,7th-8th,955,62,6.49


**Interpretation:** Найвища частка доходу `>50K` спостерігається серед категорій `Prof-school` — 73.98% та `Doctorate` — 72.56%.

Для `Masters` частка становить 54.91%, а для `Bachelors` — 41.28%. У категоріях з нижчим рівнем освіти частка доходу `>50K` значно менша.

Отже, у цьому датасеті вищий рівень освіти загалом пов'язаний з більшою часткою людей з доходом понад 50 тисяч доларів на рік.

### Запит 3. Дохід серед людей з повним робочим навантаженням

**Business-question:** Як розподіляється дохід серед людей, які працюють від 40 до 60 годин на тиждень?

In [70]:
query_3 = """
SELECT
    income,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_adult
WHERE hours_per_week BETWEEN 40 AND 60
GROUP BY income
ORDER BY n_rows DESC;
"""

pd.read_sql(query_3, engine)

,income,n_rows,pct_rows
0,<=50K,25486,71.83
1,>50K,9993,28.17


**Interpretation:** Серед людей, які працюють від 40 до 60 годин на тиждень, 71.83% мають дохід `<=50K`, а 28.17% — `>50K`.

Частка людей з доходом `>50K` у цій групі трохи вища, ніж у датасеті загалом, де вона становить 23.93%. Отже, у межах цього набору даних повніше робоче навантаження пов'язане з дещо більшою часткою високого доходу.

### Запит 4. Дохід серед записів з невідомою професією

**Business-question:** Як розподіляється дохід серед людей, для яких значення `occupation` відсутнє?

In [71]:
query_4 = """
SELECT
    income,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_adult
WHERE occupation IS NULL
GROUP BY income
ORDER BY n_rows DESC;
"""

pd.read_sql(query_4, engine)

,income,n_rows,pct_rows
0,<=50K,2544,90.57
1,>50K,265,9.43


**Interpretation:** Серед записів, де професія (`occupation`) відсутня, 90.57% мають дохід `<=50K`, а лише 9.43% — `>50K`.

Це значно нижча частка високого доходу, ніж у датасеті загалом, де вона становить 23.93%. Отже, записи з відсутнім значенням `occupation` мають помітно інший розподіл доходу, тому такі пропуски не варто ігнорувати під час подальшого аналізу або моделювання.

### Запит 5. Високий дохід у managerial / specialty професіях

**Business-question:** Скільки людей віком від 30 до 60 років у професіях, назва яких містить `managerial` або `specialty`, мають дохід `>50K`?

In [72]:
query_5 = """
SELECT
    occupation,
    COUNT(*) AS n_rows
FROM hw3_adult
WHERE age BETWEEN 30 AND 60
  AND occupation IS NOT NULL
  AND (
        occupation ILIKE '%%managerial%%'
        OR occupation ILIKE '%%specialty%%'
      )
  AND NOT (income = '<=50K')
GROUP BY occupation
ORDER BY n_rows DESC;
"""

pd.read_sql(query_5, engine)

,occupation,n_rows
0,Exec-managerial,2576
1,Prof-specialty,2419


**Interpretation:** Серед людей віком від 30 до 60 років з доходом `>50K` у вибраних професійних категоріях частіше зустрічається `Exec-managerial` — 2 576 записів.

Для `Prof-specialty` таких записів 2 419. Отже, в межах цієї вибірки обидві професійні групи широко представлені серед людей з високим доходом, причому `Exec-managerial` трапляється трохи частіше.

### Запит 6. Найпоширеніші країни походження за межами США

**Business-question:** Які країни походження, відмінні від `United-States`, найчастіше зустрічаються в датасеті?

In [73]:
query_6 = """
SELECT
    COALESCE(native_country, 'Unknown') AS native_country,
    COUNT(*) AS n_rows
FROM hw3_adult
WHERE native_country IS DISTINCT FROM 'United-States'
GROUP BY COALESCE(native_country, 'Unknown')
ORDER BY n_rows DESC
LIMIT 10;
"""

pd.read_sql(query_6, engine)

,native_country,n_rows
0,Mexico,951
1,Unknown,857
2,Philippines,295
3,Germany,206
4,Puerto-Rico,184
5,Canada,182
6,El-Salvador,155
7,India,151
8,Cuba,138
9,England,127


**Interpretation:** Серед записів з країною походження, відмінною від `United-States`, найчастіше зустрічається `Mexico` — 951 запис.

Ще 857 записів мають відсутнє значення `native_country` і тому відображені як `Unknown`. Серед інших країн найчастіше представлені `Philippines` — 295, `Germany` — 206 та `Puerto-Rico` — 184 записи.

### Запит 7. Унікальні професії серед людей з високим доходом

**Business-question:** Які професії представлені серед людей з доходом `>50K`?

In [74]:
query_7 = """
SELECT DISTINCT
    occupation
FROM hw3_adult
WHERE income = '>50K'
  AND occupation IS NOT NULL
ORDER BY occupation;
"""

pd.read_sql(query_7, engine)

,occupation
0,Adm-clerical
1,Armed-Forces
2,Craft-repair
3,Exec-managerial
4,Farming-fishing
5,Handlers-cleaners
6,Machine-op-inspct
7,Other-service
8,Priv-house-serv
9,Prof-specialty


**Interpretation:** Серед людей з доходом `>50K` представлені 14 різних професійних категорій.

До них належать, зокрема, `Exec-managerial`, `Prof-specialty`, `Sales`, `Tech-support`, `Craft-repair` та інші. Отже, високий дохід у датасеті не обмежується однією професійною групою, а зустрічається в різних типах зайнятості.

### Запит 8. Дохід серед людей з вищою освітою

**Business-question:** Як розподіляється дохід серед людей з рівнем освіти `Bachelors`, `Masters` або `Doctorate`?

In [75]:
query_8 = """
SELECT
    education,
    income,
    COUNT(*) AS n_rows
FROM hw3_adult
WHERE education IN ('Bachelors', 'Masters', 'Doctorate')
GROUP BY education, income
ORDER BY education, n_rows DESC;
"""

pd.read_sql(query_8, engine)

,education,income,n_rows
0,Bachelors,<=50K,4712
1,Bachelors,>50K,3313
2,Doctorate,>50K,431
3,Doctorate,<=50K,163
4,Masters,>50K,1459
5,Masters,<=50K,1198


**Interpretation:** Серед людей з освітою `Bachelors` більшість мають дохід `<=50K` — 4 712 записів проти 3 313 з доходом `>50K`.

Для `Masters` ситуація змінюється: 1 459 людей мають дохід `>50K`, а 1 198 — `<=50K`. Найбільша перевага високого доходу спостерігається серед `Doctorate`: 431 запис `>50K` проти 163 `<=50K`.

Отже, у вибраних категоріях зі зростанням рівня освіти частка людей з доходом `>50K` збільшується.

### Запит 9. Найбільші значення capital gain

**Business-question:** Які записи мають найбільші значення `capital_gain` серед людей, для яких цей показник більший за нуль?

In [76]:
query_9 = """
SELECT
    age,
    education,
    occupation,
    capital_gain,
    income
FROM hw3_adult
WHERE capital_gain > 0
ORDER BY capital_gain DESC
LIMIT 10;
"""

pd.read_sql(query_9, engine)

,age,education,occupation,capital_gain,income
0,22,HS-grad,Prof-specialty,99999,>50K
1,50,Some-college,Exec-managerial,99999,>50K
2,49,Bachelors,Prof-specialty,99999,>50K
3,52,HS-grad,Exec-managerial,99999,>50K
4,53,HS-grad,Sales,99999,>50K
5,52,Bachelors,Exec-managerial,99999,>50K
6,46,Prof-school,Prof-specialty,99999,>50K
7,43,Bachelors,Exec-managerial,99999,>50K
8,54,Prof-school,Prof-specialty,99999,>50K
9,43,Some-college,Sales,99999,>50K


**Interpretation:** У всіх 10 записах з найбільшим `capital_gain` значення дорівнює 99 999.

Усі ці люди належать до категорії доходу `>50K`, хоча вони відрізняються за віком, освітою та професією. Отже, дуже високий `capital_gain` у цьому датасеті пов’язаний з високою категорією доходу.

### Запит 10. Порівняння віку та робочого навантаження за рівнем доходу

**Business-question:** Чим відрізняються середній вік і середня кількість робочих годин на тиждень між групами доходу `<=50K` та `>50K`?

In [77]:
query_10 = """
SELECT
    income,
    COUNT(*) AS n_rows,
    ROUND(AVG(age)::NUMERIC, 2) AS avg_age,
    ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_hours_per_week
FROM hw3_adult
GROUP BY income
ORDER BY income;
"""

pd.read_sql(query_10, engine)

,income,n_rows,avg_age,avg_hours_per_week
0,<=50K,37155,36.87,38.84
1,>50K,11687,44.28,45.45


**Interpretation:** Люди з доходом `>50K` у середньому старші та працюють більше годин на тиждень.

Для групи `<=50K` середній вік становить 36.87 року, а середня кількість робочих годин — 38.84 на тиждень. Для групи `>50K` ці показники становлять 44.28 року та 45.45 години відповідно.

Отже, у цьому датасеті вища категорія доходу пов'язана як зі старшим середнім віком, так і з більшою кількістю робочих годин.

## 6. Reflection

Під час перевірки якості даних було виявлено кілька проблем. У колонках `workclass`, `occupation` та `native_country` пропуски були представлені як SQL NULL і спеціальним маркером `?`, тому під час очищення обидва варіанти були приведені до NULL. Також було знайдено 29 повністю однакових записів за всіма змістовними колонками, однак через відсутність унікального ідентифікатора людини їх не можна однозначно вважати помилковими дублікатами. У `income` виявились варіанти значень із крапкою в кінці (`<=50K.` та `>50K.`), які були нормалізовані до двох категорій. Значення віку до 90 років є статистично рідкісними, але допустимими, тому вони залишені без змін.

Для ML-моделі можна використовувати `age`, `education`, `workclass`, `occupation`, `hours_per_week`, `capital_gain`, `capital_loss`, `marital_status` та інші ознаки після відповідного кодування. `income` є цільовою змінною і не може використовуватися як feature. `fnlwgt` я б використовувала обережно, оскільки це вага запису в переписі, а не звичайна характеристика людини.

Явного leakage у цьому датасеті не виявлено, оскільки немає ознак, які стають відомими лише після визначення цільового доходу.

Для покращення pipeline варто стандартизувати представлення пропусків, уніфікувати категоріальні значення та виконувати автоматичні domain-checks ще під час завантаження.

Природна ML-задача для цього датасету — binary classification: прогнозування, чи належить дохід людини до категорії `>50K`.